In [1]:
from tqdm import tqdm
from models import tokenizer, sentiment, TopicAnalyzer
from data import df
import pandas as pd

tqdm.pandas() # funkcja umożliwiająca użycie paska postępu w metodzie apply

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
df["token_count"] = df["lyrics"].apply(
    lambda x: len(tokenizer.tokenize(x))
)

df.head()

,year,rank,artist,song,lyrics,token_count
0,1950,1,Fats Domino,The Fat Man,"They call, they call me the fat man 'Cause I w...",182
1,1950,2,Percy Mayfield,Please Send Me Someone To Love,"Understanding and peace of mind But, if it's n...",281
2,1950,3,Ruth Brown,Teardrops From My Eyes,I think of you And that's the time I feel so b...,201
3,1950,4,Nat King Cole,Mona Lisa,"Mona Lisa, Mona Lisa, men have named you You'r...",185
4,1950,5,Patti Page,Tennessee Waltz,When an old friend I happened to see I Introdu...,143


In [ ]:
sentiment_results = (
    df["lyrics"]
    .progress_apply(sentiment)
    .apply(pd.Series)
)

df[
    [
        "sentiment",
        "sentiment_negative",
        "sentiment_neutral",
        "sentiment_positive"
    ]
] = sentiment_results.rename(
    columns={
        "negative": "sentiment_negative",
        "neutral": "sentiment_neutral",
        "positive": "sentiment_positive"
    }
)

df.head()

100%|██████████| 679/679 [06:16<00:00,  1.80it/s]


,year,rank,artist,song,lyrics,sentiment_negative,sentiment_neutral,sentiment_positive,sentiment
0,1950,1,Fats Domino,The Fat Man,"They call, they call me the fat man 'Cause I w...",0.030819,0.346953,0.622227,0.591408
1,1950,2,Percy Mayfield,Please Send Me Someone To Love,"Understanding and peace of mind But, if it's n...",0.150122,0.383249,0.466629,0.316507
2,1950,3,Ruth Brown,Teardrops From My Eyes,I think of you And that's the time I feel so b...,0.065639,0.412966,0.521395,0.455756
3,1950,4,Nat King Cole,Mona Lisa,"Mona Lisa, Mona Lisa, men have named you You'r...",0.402747,0.504531,0.092723,-0.310024
4,1950,5,Patti Page,Tennessee Waltz,When an old friend I happened to see I Introdu...,0.617004,0.320101,0.062896,-0.554108


In [5]:
topic_analyzer = TopicAnalyzer()

topics, probs = topic_analyzer.fit(df["lyrics"].tolist())

df["topic"] = topics

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (390 > 384). Running this sequence through the model will result in indexing errors


Batches:   0%|          | 0/37 [00:00<?, ?it/s]

2026-08-10 20:12:42,827 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-10 20:13:07,992 - BERTopic - Dimensionality - Completed ✓
2026-08-10 20:13:07,994 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-10 20:13:08,141 - BERTopic - Cluster - Completed ✓
2026-08-10 20:13:08,153 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-10 20:13:08,522 - BERTopic - Representation - Completed ✓


In [6]:
topic_info = topic_analyzer.get_topic_info()
print(topic_info)

   Topic  Count                                           Name  \
0     -1    112                  -1_united_shake shake_ayy_que   
1      0    186                0_way love_really got_thank_bye   
2      1    139                    1_ayy_bitch_jump jump_nigga   
3      2     83          2_free fallin_west_fallin free_jungle   
4      3     44            3_shack_johnny_red light_microphone   
5      4     38             4_hole_prayer_sun come_dream dream   
6      5     34             5_celebrate_night night_pump_freak   
7      6     23           6_run away_upside_let start_day just   
8      7     20  7_wanna dance_let dance_dance dance_dance let   

                                      Representation  \
0  [united, shake shake, ayy, que, tu, ayy ayy, b...   
1  [way love, really got, thank, bye, love life, ...   
2  [ayy, bitch, jump jump, nigga, boom boom, boog...   
3  [free fallin, west, fallin free, jungle, purpl...   
4  [shack, johnny, red light, microphone, midnigh...   
5  

In [7]:
topic_analyzer.visualize_topics()

In [8]:
outliers = df[df["topic"] == -1]
outliers[["artist", "song", "lyrics"]].head(10)

df.head()

,year,rank,artist,song,lyrics,lyrics_topics,token_count,sentiment,topic_token_count,topic
0,1950,1,Fats Domino,The Fat Man,"They call, they call me the fat man 'Cause I w...",they call they call me the fat man cause i wei...,182,0.591408,157,1
1,1950,2,Percy Mayfield,Please Send Me Someone To Love,"Understanding and peace of mind But, if it's n...",understanding and peace of mind but if it s no...,281,0.316507,294,0
2,1950,3,Ruth Brown,Teardrops From My Eyes,I think of you And that's the time I feel so b...,i think of you and that s the time i feel so b...,201,0.455756,210,0
3,1950,4,Nat King Cole,Mona Lisa,"Mona Lisa, Mona Lisa, men have named you You'r...",mona lisa mona lisa men have named you you re ...,185,-0.310024,180,0
4,1950,5,Patti Page,Tennessee Waltz,When an old friend I happened to see I Introdu...,when an old friend i happened to see i introdu...,143,-0.554108,136,7


In [9]:
df["year"] = pd.to_numeric(df["year"], errors="coerce")

df_time = df.dropna(subset=["year"]).copy()
df_time["year"] = df_time["year"].astype(int)

In [10]:
topics_over_time = topic_analyzer.model.topics_over_time(
    docs=df_time["lyrics"].tolist(),
    timestamps=df_time["year"].tolist()
)

70it [00:03, 21.99it/s]


In [11]:
fig = topic_analyzer.model.visualize_topics_over_time(
    topics_over_time
)

fig.show()

In [12]:
df.to_excel("results.xlsx", index=False)